# Music Generator - Interactive Exploration

This notebook provides an interactive environment to explore music generation using PCA and SVD.

## Overview
- Load and visualize music files
- Perform PCA/SVD decomposition
- Analyze principal components
- Generate new music by blending components
- Listen to results

In [ ]:
# Import libraries
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from music_generator import MusicGenerator
from decomposer import MusicDecomposer
from audio_processor import AudioProcessor
from visualizer import *

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Imports successful!")

## 1. Load Music Files

First, let's load some music files and convert them to spectrograms.

In [ ]:
# Initialize processor
processor = AudioProcessor(sample_rate=22050)

# Load a sample audio file (update path as needed)
audio_file = '../data/input/sample.wav'

# Check if file exists
if Path(audio_file).exists():
    audio = processor.load_audio(audio_file, duration=30)  # Load first 30 seconds
    print(f"Loaded audio: {len(audio)} samples, {len(audio)/processor.sample_rate:.2f} seconds")
    
    # Play the audio
    display(Audio(audio, rate=processor.sample_rate))
else:
    print(f"⚠️  File not found: {audio_file}")
    print("Please add audio files to ../data/input/")

## 2. Convert to Spectrogram

In [ ]:
# Convert to spectrogram
if Path(audio_file).exists():
    spectrogram = processor.audio_to_spectrogram(audio, mel=True)
    print(f"Spectrogram shape: {spectrogram.shape}")
    print(f"Time steps: {spectrogram.shape[1]}, Frequency bins: {spectrogram.shape[0]}")
    
    # Visualize
    processor.visualize_spectrogram(spectrogram, title="Mel Spectrogram")

## 3. Perform PCA Decomposition

In [ ]:
# Initialize decomposer
decomposer = MusicDecomposer(method='pca')

# Decompose
if Path(audio_file).exists():
    n_components = 20
    components = decomposer.decompose_pca(spectrogram.T, n_components=n_components)
    
    print(f"Components shape: {components['components'].shape}")
    print(f"Transformed shape: {components['transformed'].shape}")
    print(f"\nVariance explained by top 5 components:")
    for i, var in enumerate(decomposer.variance_explained[:5], 1):
        print(f"  Component {i}: {var:.2%}")

## 4. Visualize Components

In [ ]:
# Plot variance explained
if Path(audio_file).exists():
    decomposer.plot_variance_explained()
    plot_component_contribution(decomposer.variance_explained, n_display=20)

## 5. Reconstruct Audio

Reconstruct audio using different numbers of components to see the effect.

In [ ]:
# Reconstruct with different component counts
if Path(audio_file).exists():
    for n in [5, 10, 20]:
        print(f"\nReconstruction with {n} components:")
        
        # Reconstruct spectrogram
        reconstructed = decomposer.reconstruct_pca(components, n_components=n)
        reconstructed_spec = reconstructed.T
        
        # Convert back to audio
        reconstructed_audio = processor.spectrogram_to_audio(reconstructed_spec, mel=True)
        
        # Play reconstructed audio
        print(f"Playing reconstruction with {n} components:")
        display(Audio(reconstructed_audio, rate=processor.sample_rate))

## 6. Full Pipeline - Blend Multiple Songs

In [ ]:
# Use the high-level MusicGenerator class
generator = MusicGenerator(method='pca', sample_rate=22050)

# Load multiple songs (update paths)
song_files = [
    '../data/input/song1.wav',
    '../data/input/song2.wav',
]

# Check which files exist
existing = [f for f in song_files if Path(f).exists()]

if len(existing) >= 2:
    # Load and decompose
    generator.load_songs(existing)
    generator.decompose(n_components=15)
    
    # Generate with custom weights
    weights = [0.6, 0.4]  # Favor first song
    plot_blending_weights(weights, [Path(f).stem for f in existing])
    
    new_music = generator.generate(weights=weights)
    
    # Save and play
    output_path = '../data/output/notebook_blend.wav'
    generator.save(output_path)
    
    # Load and play the result
    result_audio = processor.load_audio(output_path)
    display(Audio(result_audio, rate=processor.sample_rate))
else:
    print("⚠️  Need at least 2 audio files for blending")
    print("Add files to ../data/input/")

## 7. Experiment Section

Use this section to experiment with different parameters and approaches.

In [ ]:
# Your experiments here!
# Try:
# - Different numbers of components
# - Different blending weights
# - SVD vs PCA
# - Different audio representations (MFCC, chromagram)
